# MANTA - Non-Rigid Alignment

This notebook illustrates the novel paradigm implemented in MANTA. 

### The Problem

Originally, MANTA was engineered as a pipeline where information flows from one *module* to the next. The steps were as follows:

1. Preprocessing
2. Rigid alignment
3. Sampling
4. Feature engineering
5. Embedding
6. Matching
7. Non-rigid registration

The problem with an architecture like this is that any of these modules can fail, leading to an inconsistent or unusable result. 

### The Solution

MANTA has now been reworked to also support `manta.align.registration`, a function that exhibits the same functionality as `manta.align.non_rigid`. In contrast to the pipeline architecture used previously, MANTA will here use an integrated architecture that

1. Computes a *canonical field* $\Psi : \mathbb{R}^D \to \mathbb{R}^d$, a multi-resolution hash-encoded neural field that maps any point in the shared canonical frame to a latent expression code. $\Psi$ is the object all slices are registered to, and it is never pre-computed. Instead, it is learned jointly with everything else.
2. Learns a per-slice deformation $\psi_k$ as a multi-scale cubic B-spline free-form deformation (like before), parametrized by control point means $\mu_k^{s}$ and diagonal variances $v_k^{s}$ under a Gaussian smoothness prior. Each scale $s$ is a residual applied on top of the previous, and per-point responsibilities $r_{k, i}$ down-weight unmatched regions (to allow for partial overlap, growth and tears to be handled implicitly).
3. Trains an expression encoder $f_{\theta}$ using SSL on raw expression alone (or pre-engineered features, based on user input). This anchors the latent space, much like the embedding in the previous version.
4. Solves unbalanced entropic OT on the current embeddings to produce displacement targets $\delta_{k, i}$ from barycentric projections. These matches are still the driver for the CAVI updates of $\psi_k$.

The result is **one model** where the canonical field, embedding and deformations co-adapt to each other. Each component has a well-posed objective with no degenerate optimum (i.e. representation collapse). The API remains unchanged for other methods and previous alignment methods still work as intented.

In [1]:
import anndata as ad
import manta
import numpy as np

# I/O
from pathlib import Path

In [2]:
%load_ext autoreload
%autoreload 2

## Preprocessing

In [3]:
data_dir = Path("/staging/leuven/stg_00002/lcb/lcb_projects/SPF/mkovacic/data/dataset_MOUSE_BRAIN")

In [4]:
source = ad.read_h5ad(data_dir / "slice_75.h5ad")
target = ad.read_h5ad(data_dir / "slice_76.h5ad")

In [5]:
adatas = manta.preprocessing.preprocess(
    source=source,
    target=target,
    gene_key="gene_name",
    spatial_key="X_spatial_coords",
    max_harmony_iterations=25
)

Preprocessing:   0%|          | 0/6 [00:00<?, ?it/s]

In [6]:
source = adatas[0]
target = adatas[1]

## Rigid Alignment

In [7]:
manta.align.rigid(
    source,
    target,
    voxel_scales=[2500, 1000, 500, 250, 125],
    spatial_key="spatial_manta",
    expression_key="X_pca"
)

Rigid Alignment:   0%|          | 0/6 [00:00<?, ?it/s]

## Sampling

In [8]:
manta.sampling.sample(
    adatas=adatas,
    sampler="importance",
    fraction=0.5,
    bin_size=32,
    spatial_key="rigid"
)

## Non-rigid Registration

In [9]:
manta.align.register(
    slices=[source, target],
    spatial_key="rigid",
    expression_key="X_pca",
    inter_slice_distance=50.0
)


=== Round 1/3 ===
[SSL epoch 0] loss = 10.7951
[SSL epoch 1] loss = 6.5965
[SSL epoch 2] loss = 4.9721
[SSL epoch 3] loss = 4.1790
[SSL epoch 4] loss = 3.8047
[SSL epoch 5] loss = 3.6038
[SSL epoch 6] loss = 3.5208
[SSL epoch 7] loss = 3.4216
[SSL epoch 8] loss = 3.3970
[SSL epoch 9] loss = 3.3701
[Scale 0] grid=(8, 8, 8)
    Computing displacement targets
    Running JEPA <> CAVI iterations
w: torch.Size([8192, 64])
idx: torch.Size([8192, 64])
v: torch.Size([512])
v[idx]: torch.Size([8192, 64])
w: torch.Size([8192, 64])
idx: torch.Size([8192, 64])
v: torch.Size([512])
v[idx]: torch.Size([8192, 64])
w: torch.Size([8192, 64])
idx: torch.Size([8192, 64])
v: torch.Size([512])
v[idx]: torch.Size([8192, 64])
w: torch.Size([8192, 64])
idx: torch.Size([8192, 64])
v: torch.Size([512])
v[idx]: torch.Size([8192, 64])
w: torch.Size([8192, 64])
idx: torch.Size([8192, 64])
v: torch.Size([512])
v[idx]: torch.Size([8192, 64])
w: torch.Size([4283, 64])
idx: torch.Size([4283, 64])
v: torch.Size([512])

TypeError: logaddexp(): argument 'other' (position 2) must be Tensor, not float